<h1>The Problem

A telecom company wants to identify customers who are likely to churn. Meaning they dip

They collected the following data:

In [2]:
data = {
    "age": [22, 45, 31, 52, 28, 41, 36, 60, 25, 48,
            33, 55, 29, 39, 44, 26, 51, 34, 42, 30],

    "monthly_charge": [40, 85, 60, 120, 55, 95, 75, 140, 45, 110,
                       65, 130, 50, 80, 100, 48, 125, 70, 90, 58],

    "months_with_company": [3, 48, 12, 60, 8, 36, 24, 72, 5, 54,
                            18, 66, 6, 30, 42, 9, 57, 20, 39, 15],

    "support_calls": [5, 1, 4, 0, 6, 2, 3, 0, 7, 1,
                      5, 0, 6, 2, 1, 5, 0, 4, 2, 3],

    "satisfaction": [2, 5, 3, 5, 2, 4, 3, 5, 1, 4,
                     2, 5, 1, 4, 5, 2, 5, 3, 4, 3],

    "churn": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
              1, 0, 1, 0, 0, 1, 0, 1, 0, 1]
}

0 = Customer stayed
1 = Customer churned

Your objective

Build a model that predicts:

Will this customer churn?

1. Prepare raw data
2. Split data
3. Choose models
4. Pipelines and preprocessing
5. Cross validation. Normal or stratified?
6. Tuning
7. Evaluation

In [3]:
#Step 1 Data
import pandas as pd

df = pd.DataFrame(data)

df.head()

,age,monthly_charge,months_with_company,support_calls,satisfaction,churn
0,22,40,3,5,2,1
1,45,85,48,1,5,0
2,31,60,12,4,3,1
3,52,120,60,0,5,0
4,28,55,8,6,2,1


In [4]:
# Features and target
X = df[["age", "monthly_charge", "months_with_company", "support_calls", "satisfaction"]]
y = df["churn"]


In [5]:
#Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state =42
)

What models? Since this is a classification problem, let's use KNN, RF and LR

In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

#So only LR and KNN  need scaling, but we'll use pipelines for the models then use either grid or randomized search

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
rf_pipeline = Pipeline([
    ("rf_model", RandomForestClassifier( random_state = 42))
])

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr_model", LogisticRegression())
])
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn_model", KNeighborsClassifier())
])

In [8]:
#So we have 3 models, let us use randomized search for all
#Keep in mind that the parameter names have to point to the model names in the pipeline with double underscore
rf_param = {
    "rf_model__n_estimators" : [10, 50, 100],
    "rf_model__max_depth" : [2, 3, 5, None ]
}
knn_param = {
    "knn_model__n_neighbors": [3, 5, 7, 9]
}
lr_param = {
    "lr_model__C": [0.01, 0.1, 1, 10, 100],
    "lr_model__class_weight": [None, "balanced"]
}

In [9]:
#Let's use stratified K fold instead of classic cv
from sklearn.model_selection import StratifiedKFold

In [10]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

so in our grid or randomized search, instead of cv =5, we say cv = skf

Let's use Randomized search for all 3

In [11]:
from sklearn.model_selection import RandomizedSearchCV

In [12]:
rf_random = RandomizedSearchCV(
    rf_pipeline,
    n_iter = 10,
    param_distributions = rf_param,
    cv = skf,
    scoring = "recall",
    random_state = 42
    
)

In [13]:
#Time to fit
rf_random.fit(X_train, y_train)
#so rf_random checks the pipeline, see it's a RF, so it takes the pipeline as an estimator,
#it takes 10 combos using the param_dist, sees that CV is startified


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('rf_model',
                                              RandomForestClassifier(random_state=42))]),
                   param_distributions={'rf_model__max_depth': [2, 3, 5, None],
                                        'rf_model__n_estimators': [10, 50,
                                                                   100]},
                   random_state=42, scoring='recall')

<h3>but why did we take scoring as recall?

Earlier, the business said:

"Our biggest concern is failing to identify customers who are genuinely going to churn."

That means false negatives matter.

So we use recall instead of accuracy, recall tells us how many right answers the model found

In [14]:
knn_random = RandomizedSearchCV(
    knn_pipeline,
    n_iter = 10,
    param_distributions = knn_param,
    cv = skf,
    scoring = "recall",
    random_state = 42
    
)

In [15]:
knn_random.fit(X_train, y_train)

C:\Users\HP\miniconda3\envs\fyp\Lib\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 4 is smaller than n_iter=10. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('knn_model',
                                              KNeighborsClassifier())]),
                   param_distributions={'knn_model__n_neighbors': [3, 5, 7, 9]},
                   random_state=42, scoring='recall')

In [16]:
lr_random = RandomizedSearchCV(
    lr_pipeline,
    n_iter = 10,
    param_distributions = lr_param,
    cv = skf,
    scoring = "recall",
    random_state = 42
    
)

In [17]:
lr_random.fit(X_train, y_train)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('lr_model',
                                              LogisticRegression())]),
                   param_distributions={'lr_model__C': [0.01, 0.1, 1, 10, 100],
                                        'lr_model__class_weight': [None,
                                                                   'balanced']},
                   random_state=42, scoring='recall')

<h3>All our models have been fitted. Let's evaluate

In [18]:
best_rf = rf_random.best_estimator_
best_knn = knn_random.best_estimator_
best_lr = lr_random.best_estimator_

In [19]:
print(best_rf)
print(best_knn)
print(best_lr)

Pipeline(steps=[('rf_model',
                 RandomForestClassifier(n_estimators=50, random_state=42))])
Pipeline(steps=[('scaler', StandardScaler()),
                ('knn_model', KNeighborsClassifier(n_neighbors=3))])
Pipeline(steps=[('scaler', StandardScaler()),
                ('lr_model',
                 LogisticRegression(C=0.01, class_weight='balanced'))])


<h3>Predictions

In [20]:
rf_predictions = rf_random.predict(X_test)
knn_predictions = knn_random.predict(X_test)
lr_predictions = rf_random.predict(X_test)

In [21]:
print("Actual:", y_test.to_numpy())
print("Random Forest:", rf_predictions)
print("KNN:", knn_predictions)
print("Logistic Regression:", lr_predictions)

Actual: [1 1 1 0]
Random Forest: [1 1 1 0]
KNN: [1 1 1 0]
Logistic Regression: [1 1 1 0]


<h3>Evaluation

In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

Would be stressful evalauting one by one, let's use a function

In [23]:
#reacll that the evaluation metrics take in 2 values, actual and predicted, so we define a function that takes in 3 parameters
#The model name, the actual value and the models's predictions
def evaluate_model(name, y_true, predictions):
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions, zero_division=0)
    recall = recall_score(y_true, predictions, zero_division=0)
    f1 = f1_score(y_true, predictions, zero_division=0)

    print(f"\n{name}")
    print(f"Accuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-score:  {f1:.3f}")

In [24]:
evaluate_model("Random Forest", y_test, rf_predictions)
evaluate_model("KNN", y_test, knn_predictions)
evaluate_model("Logistic Regression", y_test, lr_predictions)


Random Forest
Accuracy:  1.000
Precision: 1.000
Recall:    1.000
F1-score:  1.000

KNN
Accuracy:  1.000
Precision: 1.000
Recall:    1.000
F1-score:  1.000

Logistic Regression
Accuracy:  1.000
Precision: 1.000
Recall:    1.000
F1-score:  1.000


Looks perfect, it's a small data set so no wonder!

In [25]:
#We can even make a comparison table using pandas
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "KNN",
        "Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, rf_predictions),
        accuracy_score(y_test, knn_predictions),
        accuracy_score(y_test, lr_predictions)
    ],
    "Precision": [
        precision_score(y_test, rf_predictions, zero_division=0),
        precision_score(y_test, knn_predictions, zero_division=0),
        precision_score(y_test, lr_predictions, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, rf_predictions, zero_division=0),
        recall_score(y_test, knn_predictions, zero_division=0),
        recall_score(y_test, lr_predictions, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, rf_predictions, zero_division=0),
        f1_score(y_test, knn_predictions, zero_division=0),
        f1_score(y_test, lr_predictions, zero_division=0)
    ]
})

print(results)

                 Model  Accuracy  Precision  Recall   F1
0        Random Forest       1.0        1.0     1.0  1.0
1                  KNN       1.0        1.0     1.0  1.0
2  Logistic Regression       1.0        1.0     1.0  1.0


<h3>One final useful addition: confusion matrices

After evaluating, we'll want to see exactly where the models made mistakes.

You can do:

In [26]:
from sklearn.metrics import confusion_matrix

print("RF:")
print(confusion_matrix(y_test, rf_predictions))

print("\nKNN:")
print(confusion_matrix(y_test, knn_predictions))

print("\nLogistic Regression:")
print(confusion_matrix(y_test, lr_predictions))

RF:
[[1 0]
 [0 3]]

KNN:
[[1 0]
 [0 3]]

Logistic Regression:
[[1 0]
 [0 3]]


TN FP

FN TP

In [27]:
print("RF CV score:", rf_random.best_score_)
print("KNN CV score:", knn_random.best_score_)
print("LR CV score:", lr_random.best_score_)

RF CV score: 1.0
KNN CV score: 1.0
LR CV score: 1.0
